<a href="https://colab.research.google.com/github/SujalBhagat4/customer_churn_prediction/blob/main/Customer_Churn_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml

# Downloads the exact real IBM Telco Customer Churn dataset (7,043 rows)
dataset = fetch_openml(name="Telco-Customer-Churn", version=1, as_frame=True)
df = dataset.frame

print("Rows and Columns:", df.shape)
df.head()


Rows and Columns: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,'No phone service',DSL,No,Yes,No,No,No,No,Month-to-month,Yes,'Electronic check',29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,'One year',No,'Mailed check',56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,'Mailed check',53.85,108.15,Yes
3,Male,0,No,No,45,No,'No phone service',DSL,Yes,No,Yes,Yes,No,No,'One year',No,'Bank transfer (automatic)',42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,'Fiber optic',No,No,No,No,No,No,Month-to-month,Yes,'Electronic check',70.70,151.65,Yes


In [ ]:
# 1. Drop customerID if it exists
df.drop('customerID', axis=1, errors='ignore', inplace=True)

# 2. Convert TotalCharges to numeric & fill missing values with Median
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# 3. Convert target 'Churn' (Yes/No) to 1 and 0
df['Churn'] = df['Churn'].astype(str).map({'Yes': 1, 'No': 0})

# 4. Convert all text/category columns into 0s and 1s
df = pd.get_dummies(df, drop_first=True)

print("Missing values remaining:", df.isnull().sum().sum())
print("Clean shape:", df.shape)


Missing values remaining: 0
Clean shape: (7043, 31)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Separate features (X) and target answer (y)
X = df.drop('Churn', axis=1)
y = df['Churn']

# Split: 80% for training the model, 20% for testing it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the Random Forest model (max_depth=6 prevents overfitting)
model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model.fit(X_train, y_train)

print("Model training complete!")


Model training complete!


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Make predictions on the 20% unseen test customers
y_pred = model.predict(X_test)

# Print evaluation scores
print("Model Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nDetailed Report:\n", classification_report(y_test, y_pred))


Model Accuracy: 80.55 %

Confusion Matrix:
 [[955  81]
 [193 180]]

Detailed Report:
               precision    recall  f1-score   support

           0       0.83      0.92      0.87      1036
           1       0.69      0.48      0.57       373

    accuracy                           0.81      1409
   macro avg       0.76      0.70      0.72      1409
weighted avg       0.79      0.81      0.79      1409



In [ ]:
# Pick 1 customer from the test set
sample_customer = X_test.iloc[[0]]

# Ask your trained model to predict
prediction = model.predict(sample_customer)[0]
probability = model.predict_proba(sample_customer)[0][1]

if prediction == 1:
    print(f"ALERT: High Risk of Churn! (Probability: {round(probability*100, 1)}%) -> Send retention discount!")
else:
    print(f"SAFE: Customer will likely stay. (Churn risk: {round(probability*100, 1)}%)")


ALERT: High Risk of Churn! (Probability: 64.1%) -> Send retention discount!
